In [ ]:
# CPU / BLAS thread controls
# Edit CPU_AFFINITY_RANGE and/or CPU_CAP before running the rest of the notebook.
import os
import multiprocessing as mp
from pathlib import Path

CPU_AFFINITY_RANGE = (90, 110)
CPU_CAP = 14

cpu_available = mp.cpu_count()
if os.environ.get("GENTRG_SMOKE_TEST") == "1":
    CPU_CAP = min(CPU_CAP, 4)

if CPU_AFFINITY_RANGE is not None:
    cpu_start, cpu_stop = [int(v) for v in CPU_AFFINITY_RANGE]
    if cpu_stop < cpu_start:
        raise ValueError(f"CPU_AFFINITY_RANGE must satisfy stop >= start, got {CPU_AFFINITY_RANGE}")
    selected_cpus = list(range(cpu_start, cpu_stop + 1))[: int(CPU_CAP)]
else:
    selected_cpus = None

cpu_use = len(selected_cpus) if selected_cpus is not None else max(1, min(int(CPU_CAP), cpu_available))
os.environ["MY_CPU_COUNT"] = str(cpu_use)
os.environ.setdefault("MPLCONFIGDIR", str(Path("/tmp") / "mplconfig_finite_depth_gaussian_mera"))
for _var in (
    "OMP_NUM_THREADS",
    "OPENBLAS_NUM_THREADS",
    "MKL_NUM_THREADS",
    "VECLIB_MAXIMUM_THREADS",
    "NUMEXPR_NUM_THREADS",
    "NUMEXPR_MAX_THREADS",
):
    os.environ[_var] = str(cpu_use)

if selected_cpus is not None:
    try:
        os.sched_setaffinity(0, set(selected_cpus))
        print(f"CPU affinity set to cores [{min(selected_cpus)}, {max(selected_cpus)}]")
    except Exception as exc:
        print(f"CPU affinity not set: {exc}")

print(f"cpu_available={cpu_available}, cpu_use={cpu_use}, CPU_AFFINITY_RANGE={CPU_AFFINITY_RANGE}")

# Finite-Depth Gaussian MERA Attempt

This notebook tests the less restrictive prerequisite for a successful quasi-1D Gaussian MERA: independent disentanglers and isometries at each scale.  The repeated scale-invariant notebooks are kept unchanged.  Here each layer learns its own boundary unitary before extracting the layer isometry from the disentangled two-row block.

## Setup

In [ ]:
import sys
import importlib
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib import font_manager
from scipy.linalg import expm


def find_repo_root(start):
    start = Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "src" / "fgtn" / "classA_U1FGTN.py").exists():
            return candidate
    raise FileNotFoundError(f"Could not find repo root from {start}")


ROOT = find_repo_root(Path.cwd())
NOTEBOOK_DIR = ROOT / "notebooks" / "flattened_hamiltonian_analysis" / "gaussian_entanglement_rg"
SMOKE_TEST = os.environ.get("GENTRG_SMOKE_TEST") == "1"
DEEP_NY80_ENV = os.environ.get("GENTRG_DEEP_NY80") == "1"
RUN_TAG = "smoke" if SMOKE_TEST else ("deep_ny80" if DEEP_NY80_ENV else "")
DATA_DIR = NOTEBOOK_DIR / "data" / "finite_depth_mera" / RUN_TAG
FIGURES_DIR = NOTEBOOK_DIR / "figures" / "finite_depth_mera" / RUN_TAG
SRC = ROOT / "src"

for path in (DATA_DIR, FIGURES_DIR):
    path.mkdir(parents=True, exist_ok=True)

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

classA_U1FGTN_module = importlib.import_module("fgtn.classA_U1FGTN")
importlib.reload(classA_U1FGTN_module)
classA_U1FGTN = classA_U1FGTN_module.classA_U1FGTN

available_fonts = {f.name for f in font_manager.fontManager.ttflist}
if "CMU Sans Serif" in available_fonts:
    plt.rcParams["font.family"] = "CMU Sans Serif"
else:
    plt.rcParams["font.family"] = "DejaVu Sans"
    print("CMU Sans Serif not found; using DejaVu Sans")

plt.rcParams.update({
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "axes.titlesize": 9,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 7,
    "lines.linewidth": 1.3,
    "mathtext.fontset": "cm",
})
np.set_printoptions(precision=6, suppress=True)

print(f"ROOT = {ROOT}")
print(f"NOTEBOOK_DIR = {NOTEBOOK_DIR}")
print(f"DATA_DIR = {DATA_DIR}")
print(f"FIGURES_DIR = {FIGURES_DIR}")

## Run Controls

In [ ]:
NX = 20
NY = 40
NSHELL = 1
ALPHA_1 = 1.0
ALPHA_2 = 30.0
DW = True
DW_TRUNCATION = True
TRIAL_ORBITAL = "X"

BASE_FIT_AY_MIN = 8
MIN_COARSE_NY = 10
MAX_LAYERS = 2
RUN_DEEP_NY80 = os.environ.get("GENTRG_DEEP_NY80") == "1"
EDGE_CHANNEL_WIDTH = 1

ENTROPY_CLIP_EPS = 1e-12
DOMAIN_WALL_PROFILE_WIDTH = 1
R_MAX = 4
LOSS_EPS = 1e-12

RUN_ACTIVE_DISENTANGLER = True
ACTIVE_DIM = 16
RANDOM_U_STEPS = 36
RANDOM_U_TRIALS_PER_STEP = 2
RANDOM_U_INITIAL_STEP = 0.35
RANDOM_U_STEP_DECAY = 0.92
RANDOM_SEED = 1234

LAMBDA_IDEM = 1.0
LAMBDA_DISC = 4.0
LAMBDA_CROSS = 1.0
LAMBDA_FP = 0.02

if RUN_DEEP_NY80 and not SMOKE_TEST:
    NY = 80
    MAX_LAYERS = 3
    RUN_ACTIVE_DISENTANGLER = False
    RANDOM_U_STEPS = 0

if SMOKE_TEST:
    NX = 6
    NY = 16
    BASE_FIT_AY_MIN = 2
    MIN_COARSE_NY = 8
    MAX_LAYERS = 1
    R_MAX = 2
    ACTIVE_DIM = 6
    RANDOM_U_STEPS = 8
    RANDOM_U_TRIALS_PER_STEP = 1
    RANDOM_U_INITIAL_STEP = 0.4

Q_ORBITALS = 2 * NX

print(f"SMOKE_TEST = {SMOKE_TEST}")
print(f"Nx, Ny = {NX}, {NY}")
print(f"q = 2*Nx = {Q_ORBITALS}")
print(f"finite-depth layers = {MAX_LAYERS}")
print(f"RUN_DEEP_NY80 = {RUN_DEEP_NY80}")
print(f"EDGE_CHANNEL_WIDTH = {EDGE_CHANNEL_WIDTH}")
print(f"active_dim = {ACTIVE_DIM}, active optimization = {RUN_ACTIVE_DISENTANGLER}")
print(f"random optimizer steps = {RANDOM_U_STEPS}, trials/step = {RANDOM_U_TRIALS_PER_STEP}")
print(f"loss weights = idem {LAMBDA_IDEM}, disc {LAMBDA_DISC}, cross {LAMBDA_CROSS}, fp {LAMBDA_FP}")

## Target State and Entropy Helpers

In [ ]:
def build_flattened_hamiltonian(model, trial_orbital=TRIAL_ORBITAL, dw_truncation=DW_TRUNCATION):
    nlayer = 2 * model.Nx * model.Ny
    model.construct_OW_projectors(
        nshell=model.nshell,
        DW=model.DW,
        trial_orbitals=trial_orbital,
        dw_truncation=dw_truncation,
    )
    WA_p = model.WF_Ap.reshape(nlayer, -1)
    WB_p = model.WF_Bp.reshape(nlayer, -1)
    WA_m = model.WF_Am.reshape(nlayer, -1)
    WB_m = model.WF_Bm.reshape(nlayer, -1)
    H_flat = WA_p @ WA_p.conj().T + WB_p @ WB_p.conj().T - WA_m @ WA_m.conj().T - WB_m @ WB_m.conj().T
    return 0.5 * (H_flat + H_flat.conj().T)


def build_half_filled_flattened_state(nx=NX, ny=NY):
    model = classA_U1FGTN(nx, ny, nshell=NSHELL, DW=DW, alpha_1=ALPHA_1, alpha_2=ALPHA_2)
    H_flat = build_flattened_hamiltonian(model)
    evals, evecs = np.linalg.eigh(H_flat)
    nlayer = H_flat.shape[0]
    n_occ = nlayer // 2
    occ_mask = evals < 0.0
    neg_count = int(np.count_nonzero(occ_mask))
    fallback_used = False
    if neg_count != n_occ:
        occ_mask = np.zeros_like(evals, dtype=bool)
        occ_mask[np.argsort(evals)[:n_occ]] = True
        fallback_used = True
    U_occ = evecs[:, occ_mask]
    C = U_occ @ U_occ.conj().T
    C = 0.5 * (C + C.conj().T)
    return {
        "model": model,
        "H_flat": H_flat,
        "evals": np.real_if_close(evals),
        "C": C,
        "n_occ": int(n_occ),
        "neg_count": neg_count,
        "fallback_used": bool(fallback_used),
        "spectral_gap": float(np.min(np.abs(evals))),
    }


def purify_to_half_filled_projector(C, n_occ=None):
    C = 0.5 * (np.asarray(C, dtype=np.complex128) + np.asarray(C, dtype=np.complex128).conj().T)
    dim = C.shape[0]
    if n_occ is None:
        n_occ = dim // 2
    evals, evecs = np.linalg.eigh(C)
    occ = np.argsort(evals)[-int(n_occ):]
    U_occ = evecs[:, occ]
    P = U_occ @ U_occ.conj().T
    return 0.5 * (P + P.conj().T), np.real_if_close(evals)


def idempotency_defect(C):
    C = np.asarray(C, dtype=np.complex128)
    dim = C.shape[0]
    return float(np.linalg.norm(C @ C - C, ord="fro") / max(1.0, np.sqrt(dim)))


def interval_indices(n_sites, q, length, start=0):
    n_sites = int(n_sites)
    q = int(q)
    length = int(length)
    start = int(start) % n_sites
    rows = (np.arange(length, dtype=int) + start) % n_sites
    return np.concatenate([np.arange(row * q, (row + 1) * q, dtype=int) for row in rows])


def entropy_from_correlation_submatrix(C_sub, clip_eps=ENTROPY_CLIP_EPS):
    C_sub = np.asarray(C_sub, dtype=np.complex128)
    C_sub = 0.5 * (C_sub + C_sub.conj().T)
    lambdas = np.linalg.eigvalsh(C_sub)
    lambdas = np.clip(np.real_if_close(lambdas), clip_eps, 1.0 - clip_eps)
    return float(-np.sum(lambdas * np.log(lambdas) + (1.0 - lambdas) * np.log(1.0 - lambdas)))


def entropy_for_interval(C, n_sites, q, length, start=0):
    if int(length) == 0:
        return 0.0
    idx = interval_indices(n_sites, q, length, start=start)
    return entropy_from_correlation_submatrix(C[np.ix_(idx, idx)])


def mean_entropy_profile(C, n_sites, q, max_length=None):
    n_sites = int(n_sites)
    if max_length is None:
        max_length = n_sites // 2
    lengths = np.arange(1, int(max_length) + 1, dtype=int)
    values = []
    for length in lengths:
        entropies = [entropy_for_interval(C, n_sites, q, length, start=y0) for y0 in range(n_sites)]
        values.append(float(np.mean(entropies)))
    return {"Ay_values": lengths, "avg_entropy": np.asarray(values, dtype=float)}


def logchord_values(Ay_values, ny):
    Ay_values = np.asarray(Ay_values, dtype=float)
    return np.log((float(ny) / np.pi) * np.sin(np.pi * Ay_values / float(ny)))


def fit_ay_min_for_ny(ny):
    ny = int(ny)
    if ny <= 12:
        return 1
    return min(int(BASE_FIT_AY_MIN), max(1, ny // 5))


def fit_entropy_logchord(Ay_values, entropies, ny, ay_min=None):
    Ay_values = np.asarray(Ay_values, dtype=float)
    entropies = np.asarray(entropies, dtype=float)
    if ay_min is None:
        ay_min = fit_ay_min_for_ny(ny)
    mask = (Ay_values >= int(ay_min)) & (Ay_values <= int(ny) // 2)
    if int(np.count_nonzero(mask)) < 2:
        mask = Ay_values <= int(ny) // 2
    x = logchord_values(Ay_values[mask], ny)
    y = entropies[mask]
    coeff, cov = np.polyfit(x, y, deg=1, cov=True) if len(x) > 2 else (np.polyfit(x, y, deg=1), np.full((2, 2), np.nan))
    slope, intercept = [float(v) for v in coeff]
    yhat = slope * x + intercept
    ss_res = float(np.sum((y - yhat) ** 2))
    ss_tot = float(np.sum((y - np.mean(y)) ** 2)) if len(y) > 1 else np.nan
    return {
        "slope": slope,
        "intercept": intercept,
        "slope_err": float(np.sqrt(cov[0, 0])) if np.all(np.isfinite(cov)) else np.nan,
        "c_est": float(3.0 * slope),
        "c_est_err": float(3.0 * np.sqrt(cov[0, 0])) if np.all(np.isfinite(cov)) else np.nan,
        "r2": float(1.0 - ss_res / ss_tot) if ss_tot and ss_tot > 0 else np.nan,
        "ay_min": int(ay_min),
        "n_fit": int(len(x)),
    }

## Gaussian Layer Primitives

In [ ]:
def row_indices(row, q, n_sites):
    row = int(row) % int(n_sites)
    return np.arange(row * q, (row + 1) * q, dtype=int)


def pair_indices(row_a, row_b, q, n_sites):
    return np.concatenate([row_indices(row_a, q, n_sites), row_indices(row_b, q, n_sites)])


def block_pairs(n_sites):
    return [(2 * j, 2 * j + 1) for j in range(int(n_sites) // 2)]


def boundary_pairs(n_sites):
    return [(2 * j + 1, (2 * j + 2) % int(n_sites)) for j in range(int(n_sites) // 2)]


def averaged_pair_correlation(C, n_sites, q, pairs):
    acc = np.zeros((2 * q, 2 * q), dtype=np.complex128)
    for row_a, row_b in pairs:
        idx = pair_indices(row_a, row_b, q, n_sites)
        acc += C[np.ix_(idx, idx)]
    acc /= float(len(pairs))
    return 0.5 * (acc + acc.conj().T)


def natural_orbital_isometry(C_pair, q):
    C_pair = 0.5 * (np.asarray(C_pair, dtype=np.complex128) + np.asarray(C_pair, dtype=np.complex128).conj().T)
    occ, vecs = np.linalg.eigh(C_pair)
    occ = np.clip(np.real_if_close(occ), 0.0, 1.0)
    chi = occ * (1.0 - occ)
    keep = np.argsort(chi)[-int(q):]
    discard = np.setdiff1d(np.arange(2 * int(q)), keep, assume_unique=False)
    keep = keep[np.argsort(occ[keep])]
    discard = discard[np.argsort(occ[discard])]
    return {
        "w": vecs[:, keep],
        "d": vecs[:, discard],
        "kept_occ": occ[keep].copy(),
        "discarded_occ": occ[discard].copy(),
        "kept_chi": chi[keep].copy(),
        "discarded_chi": chi[discard].copy(),
        "all_occ": occ.copy(),
        "all_chi": chi.copy(),
    }


def average_block_correlator(C, n_sites, q, r):
    n_sites = int(n_sites)
    q = int(q)
    r = int(r) % n_sites
    acc = np.zeros((q, q), dtype=np.complex128)
    for y in range(n_sites):
        i = row_indices(y, q, n_sites)
        j = row_indices(y + r, q, n_sites)
        acc += C[np.ix_(i, j)]
    return acc / float(n_sites)


def fixed_point_loss(C_target, C_coarse, n_sites_target, n_sites_coarse, q, r_max=R_MAX, scale_factor=2):
    max_r = min(int(r_max), int(n_sites_coarse // 2))
    total = 0.0
    for r in range(max_r + 1):
        coarse = average_block_correlator(C_coarse, n_sites_coarse, q, r)
        target = average_block_correlator(C_target, n_sites_target, q, int(scale_factor) * r)
        numerator = float(np.linalg.norm(coarse - target, ord="fro") ** 2)
        denom = float(np.linalg.norm(target, ord="fro") ** 2 + LOSS_EPS)
        total += (1.0 / float(1 + r)) * numerator / denom
    return float(total)


def boundary_unitary_global_matrix(n_sites, q, u):
    dim = int(n_sites) * int(q)
    mat = np.eye(dim, dtype=np.complex128)
    if u is None:
        return mat
    for row_a, row_b in boundary_pairs(n_sites):
        idx = pair_indices(row_a, row_b, q, n_sites)
        mat[np.ix_(idx, idx)] = u
    return mat


def repeated_isometry_global_matrix(n_sites, q, w):
    n_sites = int(n_sites)
    q = int(q)
    out_sites = n_sites // 2
    mat = np.zeros((n_sites * q, out_sites * q), dtype=np.complex128)
    for j, (row_a, row_b) in enumerate(block_pairs(n_sites)):
        idx = pair_indices(row_a, row_b, q, n_sites)
        mat[np.ix_(idx, np.arange(j * q, (j + 1) * q))] = w
    return mat


def apply_boundary_unitary(C, n_sites, q, u):
    if u is None:
        return np.asarray(C, dtype=np.complex128)
    C_u = np.asarray(C, dtype=np.complex128).copy()
    for row_a, row_b in boundary_pairs(n_sites):
        idx = pair_indices(row_a, row_b, q, n_sites)
        C_u[idx, :] = u @ C_u[idx, :]
    for row_a, row_b in boundary_pairs(n_sites):
        idx = pair_indices(row_a, row_b, q, n_sites)
        C_u[:, idx] = C_u[:, idx] @ u.conj().T
    return 0.5 * (C_u + C_u.conj().T)


def apply_layer(C, n_sites, q, u, w):
    C_tilde = apply_boundary_unitary(C, n_sites, q, u)
    W = repeated_isometry_global_matrix(n_sites, q, w)
    C_raw = W.conj().T @ C_tilde @ W
    C_raw = 0.5 * (C_raw + C_raw.conj().T)
    C_pure, raw_evals = purify_to_half_filled_projector(C_raw)
    U_global = boundary_unitary_global_matrix(n_sites, q, u)
    embed_layer = U_global.conj().T @ W
    return C_raw, C_pure, raw_evals, embed_layer, C_tilde


def layer_local_diagnostics(C_tilde, C_raw, n_sites, q, w, data, C_reference=None, reference_n_sites=None, scale_factor=2):
    C_block = averaged_pair_correlation(C_tilde, n_sites, q, block_pairs(n_sites))
    d = data["d"]
    cross = w.conj().T @ C_block @ d
    fp = 0.0
    if C_reference is not None and reference_n_sites is not None:
        fp = fixed_point_loss(C_reference, C_raw, reference_n_sites, n_sites // 2, q, scale_factor=scale_factor)
    idem = idempotency_defect(C_raw)
    disc_mean = float(np.mean(data["discarded_chi"]))
    disc_max = float(np.max(data["discarded_chi"]))
    cross_norm = float(np.linalg.norm(cross, ord="fro") ** 2 / max(1.0, q))
    total = float(LAMBDA_IDEM * idem * idem + LAMBDA_DISC * disc_mean + LAMBDA_CROSS * cross_norm + LAMBDA_FP * fp)
    return {
        "total": total,
        "idem": float(idem),
        "idem_sq": float(idem * idem),
        "discarded_chi_mean": disc_mean,
        "discarded_chi_max": disc_max,
        "cross": cross_norm,
        "fp": float(fp),
    }


def extract_layer_isometry_after_u(C, n_sites, q, u):
    C_tilde = apply_boundary_unitary(C, n_sites, q, u)
    C_block = averaged_pair_correlation(C_tilde, n_sites, q, block_pairs(n_sites))
    data = natural_orbital_isometry(C_block, q)
    data["average_block_C"] = C_block
    return data, C_tilde

## Active Boundary Disentangler Optimizer

In [ ]:
def init_active_boundary_basis(C, n_sites, q, active_dim):
    C_boundary = averaged_pair_correlation(C, n_sites, q, boundary_pairs(n_sites))
    occ, vecs = np.linalg.eigh(C_boundary)
    occ = np.clip(np.real_if_close(occ), 0.0, 1.0)
    chi = occ * (1.0 - occ)
    active = np.argsort(chi)[-int(active_dim):]
    active = active[np.argsort(occ[active])]
    return {
        "basis": vecs[:, active],
        "active_occ": occ[active].copy(),
        "active_chi": chi[active].copy(),
        "average_boundary_C": C_boundary,
    }


def random_antihermitian(m, rng):
    z = rng.normal(size=(m, m)) + 1j * rng.normal(size=(m, m))
    K = z - z.conj().T
    norm = np.linalg.norm(K, ord="fro")
    if norm == 0:
        return K
    return K / norm


def embed_active_unitary(active_u, basis, q):
    m = basis.shape[1]
    eye = np.eye(2 * int(q), dtype=np.complex128)
    return eye + basis @ (active_u - np.eye(m, dtype=np.complex128)) @ basis.conj().T


def score_layer_choice(C, n_sites, q, u, C_reference, reference_n_sites, scale_factor):
    data, C_tilde = extract_layer_isometry_after_u(C, n_sites, q, u)
    C_raw, C_pure, raw_evals, embed_layer, _ = apply_layer(C, n_sites, q, u, data["w"])
    parts = layer_local_diagnostics(
        C_tilde,
        C_raw,
        n_sites,
        q,
        data["w"],
        data,
        C_reference=C_reference,
        reference_n_sites=reference_n_sites,
        scale_factor=scale_factor,
    )
    return parts, data, C_raw, C_pure, raw_evals, embed_layer


def optimize_random_active_u(C, n_sites, q, active_dim, C_reference, reference_n_sites, scale_factor, rng):
    active = init_active_boundary_basis(C, n_sites, q, active_dim)
    basis = active["basis"]
    m = basis.shape[1]
    active_u = np.eye(m, dtype=np.complex128)
    u_best = embed_active_unitary(active_u, basis, q)
    best_parts, best_data, best_raw, best_pure, best_evals, best_embed = score_layer_choice(
        C, n_sites, q, u_best, C_reference, reference_n_sites, scale_factor
    )
    history = [{"step": 0, "accepted": True, "step_size": 0.0, **best_parts}]
    step_size = float(RANDOM_U_INITIAL_STEP)
    for step in range(1, int(RANDOM_U_STEPS) + 1):
        accepted = False
        for _ in range(int(RANDOM_U_TRIALS_PER_STEP)):
            K = random_antihermitian(m, rng)
            trial_active_u = expm(step_size * K) @ active_u
            trial_u = embed_active_unitary(trial_active_u, basis, q)
            trial_parts, trial_data, trial_raw, trial_pure, trial_evals, trial_embed = score_layer_choice(
                C, n_sites, q, trial_u, C_reference, reference_n_sites, scale_factor
            )
            if trial_parts["total"] < best_parts["total"]:
                active_u = trial_active_u
                u_best = trial_u
                best_parts = trial_parts
                best_data = trial_data
                best_raw = trial_raw
                best_pure = trial_pure
                best_evals = trial_evals
                best_embed = trial_embed
                accepted = True
        history.append({"step": int(step), "accepted": bool(accepted), "step_size": float(step_size), **best_parts})
        step_size *= float(RANDOM_U_STEP_DECAY)
    return {
        "u": u_best,
        "data": best_data,
        "C_raw": best_raw,
        "C_pure": best_pure,
        "raw_evals": best_evals,
        "embed_layer": best_embed,
        "parts": best_parts,
        "active": active,
        "history": pd.DataFrame(history),
    }

## Diagnostics

In [ ]:
def domain_wall_x_mask(nx, dw_locs, width=DOMAIN_WALL_PROFILE_WIDTH):
    mask = np.zeros(int(nx), dtype=bool)
    for loc in dw_locs:
        for dx in range(-int(width), int(width) + 1):
            mask[(int(loc) + dx) % int(nx)] = True
    return mask


def original_x_profile_from_embedding(embedding, nx, ny):
    weights = np.sum(np.abs(embedding) ** 2, axis=1)
    weights = weights.reshape(int(ny), int(nx), 2).sum(axis=(0, 2))
    total = float(np.sum(weights))
    return weights / total if total > 0 else weights


def entanglement_spectrum(C, n_sites, q, length=None):
    if length is None:
        length = max(1, int(n_sites) // 2)
    idx = interval_indices(n_sites, q, int(length), start=0)
    vals = np.linalg.eigvalsh(0.5 * (C[np.ix_(idx, idx)] + C[np.ix_(idx, idx)].conj().T))
    return np.clip(np.real_if_close(vals), 0.0, 1.0)


def analyze_state(stage, layer, C_measure, C_raw, n_sites, q, embedding, scale_factor, layer_parts=None):
    entropy_data = mean_entropy_profile(C_measure, n_sites, q)
    fit = fit_entropy_logchord(entropy_data["Ay_values"], entropy_data["avg_entropy"], n_sites)
    x_profile = original_x_profile_from_embedding(embedding, NX, NY)
    dw_mask = domain_wall_x_mask(NX, DW_LOCS)
    fp = fixed_point_loss(C0, C_measure, NY, n_sites, q, r_max=min(R_MAX, n_sites // 2), scale_factor=scale_factor)
    row = {
        "smoke_test": bool(SMOKE_TEST),
        "stage": str(stage),
        "layer": int(layer),
        "Nx": int(NX),
        "Ny_initial": int(NY),
        "Ny_current": int(n_sites),
        "q_orbitals": int(q),
        "active_dim": int(ACTIVE_DIM),
        "fixed_point_loss": float(fp),
        "raw_idempotency_defect": idempotency_defect(C_raw),
        "measure_idempotency_defect": idempotency_defect(C_measure),
        "slope": fit["slope"],
        "slope_err": fit["slope_err"],
        "c_est": fit["c_est"],
        "c_est_err": fit["c_est_err"],
        "r2": fit["r2"],
        "domain_wall_x_weight_width1": float(np.sum(x_profile[dw_mask])),
    }
    if layer_parts is not None:
        for key, value in layer_parts.items():
            row[f"layer_{key}"] = value
    return row, entropy_data, x_profile, entanglement_spectrum(C_measure, n_sites, q)

## Build Target and Run Finite-Depth Flows

In [ ]:
state = build_half_filled_flattened_state(NX, NY)
model = state["model"]
C0 = state["C"]
DW_LOCS = list(getattr(model, "DW_loc", []))
rng = np.random.default_rng(RANDOM_SEED)

print(f"DW locations = {DW_LOCS}")
print(f"H_flat shape = {state['H_flat'].shape}")
print(f"negative eigenvalue count = {state['neg_count']}")
print(f"filled modes = {state['n_occ']}")
print(f"fallback_used = {state['fallback_used']}")
print(f"spectral_gap = {state['spectral_gap']:.6e}")
print(f"projector idempotency ||C^2-C||_F/sqrt(N) = {idempotency_defect(C0):.6e}")

In [ ]:
def run_finite_depth_flow(stage, use_active_u=False, purify_between_layers=False):
    summary_rows = []
    entropy_rows = []
    occupation_rows = []
    profile_rows = []
    spectrum_rows = []
    optimizer_rows = []
    layer_tensors = []

    C_current = C0
    C_raw_current = C0
    n_sites = int(NY)
    embedding = np.eye(C0.shape[0], dtype=np.complex128)

    row, entropy_data, x_profile, spectrum = analyze_state(stage, 0, C_current, C_raw_current, n_sites, Q_ORBITALS, embedding, 1)
    summary_rows.append(row)
    for ay, ent in zip(entropy_data["Ay_values"], entropy_data["avg_entropy"]):
        entropy_rows.append({"stage": stage, "layer": 0, "Ay": int(ay), "entropy": float(ent)})
    for x, weight in enumerate(x_profile):
        profile_rows.append({"stage": stage, "layer": 0, "x": int(x), "weight": float(weight)})
    for rank, val in enumerate(spectrum):
        spectrum_rows.append({"stage": stage, "layer": 0, "rank": int(rank), "lambda": float(val)})

    for layer in range(1, int(MAX_LAYERS) + 1):
        scale_factor = 2 ** layer
        if use_active_u and RUN_ACTIVE_DISENTANGLER:
            learned = optimize_random_active_u(
                C_current,
                n_sites,
                Q_ORBITALS,
                ACTIVE_DIM,
                C0,
                NY,
                scale_factor,
                rng,
            )
            u = learned["u"]
            data = learned["data"]
            C_raw = learned["C_raw"]
            C_pure = learned["C_pure"]
            embed_layer = learned["embed_layer"]
            layer_parts = learned["parts"]
            hist = learned["history"].copy()
            hist.insert(0, "stage", stage)
            hist.insert(1, "layer", layer)
            optimizer_rows.extend(hist.to_dict("records"))
        else:
            u = None
            data, C_tilde = extract_layer_isometry_after_u(C_current, n_sites, Q_ORBITALS, u)
            C_raw, C_pure, raw_evals, embed_layer, _ = apply_layer(C_current, n_sites, Q_ORBITALS, u, data["w"])
            layer_parts = layer_local_diagnostics(
                C_tilde,
                C_raw,
                n_sites,
                Q_ORBITALS,
                data["w"],
                data,
                C_reference=C0,
                reference_n_sites=NY,
                scale_factor=scale_factor,
            )
            optimizer_rows.append({"stage": stage, "layer": layer, "step": 0, "accepted": True, "step_size": 0.0, **layer_parts})

        tensor_record = {
            "stage": stage,
            "layer": int(layer),
            "n_sites_in": int(n_sites),
            "w": np.asarray(data["w"], dtype=np.complex128).copy(),
            "u": None if u is None else np.asarray(u, dtype=np.complex128).copy(),
            "C_input": np.asarray(C_current, dtype=np.complex128).copy(),
            "parts": dict(layer_parts),
        }

        embedding = embedding @ embed_layer
        n_sites //= 2
        C_measure = C_pure if purify_between_layers else C_raw
        tensor_record.update({
            "n_sites_out": int(n_sites),
            "C_raw": np.asarray(C_raw, dtype=np.complex128).copy(),
            "C_output": np.asarray(C_measure, dtype=np.complex128).copy(),
        })
        layer_tensors.append(tensor_record)
        row, entropy_data, x_profile, spectrum = analyze_state(stage, layer, C_measure, C_raw, n_sites, Q_ORBITALS, embedding, scale_factor, layer_parts)
        summary_rows.append(row)

        for rank, val in enumerate(data["kept_occ"]):
            occupation_rows.append({"stage": stage, "layer": layer, "sector": "kept", "rank": int(rank), "occupation": float(val), "chi": float(data["kept_chi"][rank])})
        for rank, val in enumerate(data["discarded_occ"]):
            occupation_rows.append({"stage": stage, "layer": layer, "sector": "discarded", "rank": int(rank), "occupation": float(val), "chi": float(data["discarded_chi"][rank])})
        for ay, ent in zip(entropy_data["Ay_values"], entropy_data["avg_entropy"]):
            entropy_rows.append({"stage": stage, "layer": layer, "Ay": int(ay), "entropy": float(ent)})
        for x, weight in enumerate(x_profile):
            profile_rows.append({"stage": stage, "layer": layer, "x": int(x), "weight": float(weight)})
        for rank, val in enumerate(spectrum):
            spectrum_rows.append({"stage": stage, "layer": layer, "rank": int(rank), "lambda": float(val)})

        C_current = C_measure
        C_raw_current = C_raw
        if n_sites <= MIN_COARSE_NY:
            break

    return {
        "summary": pd.DataFrame(summary_rows),
        "entropy": pd.DataFrame(entropy_rows),
        "occupations": pd.DataFrame(occupation_rows),
        "profiles": pd.DataFrame(profile_rows),
        "spectrum": pd.DataFrame(spectrum_rows),
        "optimizer": pd.DataFrame(optimizer_rows),
        "layer_tensors": layer_tensors,
    }

flows = []
flows.append(run_finite_depth_flow("finite_depth_uI_raw", use_active_u=False, purify_between_layers=False))
flows.append(run_finite_depth_flow("finite_depth_active_u_raw", use_active_u=True, purify_between_layers=False))
flows.append(run_finite_depth_flow("finite_depth_active_u_purified_next", use_active_u=True, purify_between_layers=True))

summary_df = pd.concat([f["summary"] for f in flows], ignore_index=True)
entropy_df = pd.concat([f["entropy"] for f in flows], ignore_index=True)
occupation_df = pd.concat([f["occupations"] for f in flows], ignore_index=True)
profile_df = pd.concat([f["profiles"] for f in flows], ignore_index=True)
spectrum_df = pd.concat([f["spectrum"] for f in flows], ignore_index=True)
optimizer_df = pd.concat([f["optimizer"] for f in flows], ignore_index=True)

summary_path = DATA_DIR / "finite_depth_gaussian_mera_summary.csv"
entropy_path = DATA_DIR / "finite_depth_gaussian_mera_entropy.csv"
occupation_path = DATA_DIR / "finite_depth_gaussian_mera_occupations.csv"
profile_path = DATA_DIR / "finite_depth_gaussian_mera_x_profiles.csv"
spectrum_path = DATA_DIR / "finite_depth_gaussian_mera_entanglement_spectrum.csv"
optimizer_path = DATA_DIR / "finite_depth_gaussian_mera_optimizer.csv"

summary_df.to_csv(summary_path, index=False)
entropy_df.to_csv(entropy_path, index=False)
occupation_df.to_csv(occupation_path, index=False)
profile_df.to_csv(profile_path, index=False)
spectrum_df.to_csv(spectrum_path, index=False)
optimizer_df.to_csv(optimizer_path, index=False)

print(f"Saved summary: {summary_path}")
print(f"Saved optimizer history: {optimizer_path}")
summary_df

## Gauge Alignment and Scale-Invariant Fit from Finite-Depth Tensors

In [ ]:

def output_gauge_align(w_candidate, w_reference):
    overlap = w_candidate.conj().T @ w_reference
    U, _, Vh = np.linalg.svd(overlap, full_matrices=False)
    R = U @ Vh
    return w_candidate @ R, R




def fix_vector_phases(vecs):
    vecs = np.asarray(vecs, dtype=np.complex128).copy()
    for col in range(vecs.shape[1]):
        pivot = int(np.argmax(np.abs(vecs[:, col])))
        phase = np.angle(vecs[pivot, col])
        vecs[:, col] *= np.exp(-1j * phase)
        if np.real(vecs[pivot, col]) < 0:
            vecs[:, col] *= -1.0
    return vecs


def canonical_site_gauge(C, n_sites, q, degeneracy_tol=1e-7):
    C0bar = average_block_correlator(C, n_sites, q, 0)
    C1bar = average_block_correlator(C, n_sites, q, 1)
    C0bar = 0.5 * (C0bar + C0bar.conj().T)
    H1 = 0.5 * (C1bar + C1bar.conj().T)
    evals, vecs = np.linalg.eigh(C0bar)
    order = np.argsort(evals)
    evals = np.real_if_close(evals[order])
    vecs = vecs[:, order]
    refined = vecs.copy()
    start = 0
    while start < q:
        stop = start + 1
        while stop < q and abs(float(evals[stop] - evals[start])) < degeneracy_tol:
            stop += 1
        if stop - start > 1:
            sub = vecs[:, start:stop]
            hsub = sub.conj().T @ H1 @ sub
            hsub = 0.5 * (hsub + hsub.conj().T)
            hvals, hvecs = np.linalg.eigh(hsub)
            horder = np.argsort(np.real_if_close(hvals))
            refined[:, start:stop] = sub @ hvecs[:, horder]
        start = stop
    return fix_vector_phases(refined)


def block_diag_two(mat):
    q = mat.shape[0]
    out = np.zeros((2 * q, 2 * q), dtype=np.complex128)
    out[:q, :q] = mat
    out[q:, q:] = mat
    return out


def canonicalize_layer_tensor(tensor, q):
    Gin = canonical_site_gauge(tensor["C_input"], tensor["n_sites_in"], q)
    Gout = canonical_site_gauge(tensor["C_output"], tensor["n_sites_out"], q)
    w_can = block_diag_two(Gin.conj().T) @ tensor["w"] @ Gout
    return w_can, Gin, Gout

def tensor_alignment_metrics(w0, w1):
    w1_aligned, R = output_gauge_align(w1, w0)
    svals = np.linalg.svd(w0.conj().T @ w1, compute_uv=False)
    P0 = w0 @ w0.conj().T
    P1 = w1 @ w1.conj().T
    return {
        "relative_output_gauge_delta": float(np.linalg.norm(w1_aligned - w0, ord="fro") / max(1.0, np.linalg.norm(w0, ord="fro"))),
        "projector_delta_per_sqrt_q": float(np.linalg.norm(P1 - P0, ord="fro") / np.sqrt(w0.shape[1])),
        "principal_sval_min": float(np.min(svals)),
        "principal_sval_mean": float(np.mean(svals)),
        "principal_sval_max": float(np.max(svals)),
        "aligned_w1": w1_aligned,
        "output_rotation": R,
    }


def average_isometries(w0, w1):
    w1_aligned, _ = output_gauge_align(w1, w0)
    qmat, _ = np.linalg.qr(0.5 * (w0 + w1_aligned), mode="reduced")
    return qmat


def run_repeated_candidate_flow(stage, w_repeated, max_layers=MAX_LAYERS):
    rows = []
    C_current = C0
    C_raw_current = C0
    n_sites = int(NY)
    embedding = np.eye(C0.shape[0], dtype=np.complex128)
    row, entropy_data, x_profile, spectrum = analyze_state(stage, 0, C_current, C_raw_current, n_sites, Q_ORBITALS, embedding, 1)
    rows.append(row)
    for layer in range(1, int(max_layers) + 1):
        scale_factor = 2 ** layer
        C_raw, C_pure, raw_evals, embed_layer, C_tilde = apply_layer(C_current, n_sites, Q_ORBITALS, None, w_repeated)
        embedding = embedding @ embed_layer
        n_sites //= 2
        C_current = C_raw
        row, entropy_data, x_profile, spectrum = analyze_state(stage, layer, C_current, C_raw, n_sites, Q_ORBITALS, embedding, scale_factor)
        rows.append(row)
        if n_sites <= MIN_COARSE_NY:
            break
    return pd.DataFrame(rows)

alignment_rows = []
canonical_alignment_rows = []
period2_alignment_rows = []
fitted_rows = []
for flow in flows:
    tensors = {item["layer"]: item for item in flow["layer_tensors"]}
    stage = flow["summary"]["stage"].iloc[0]
    if 1 not in tensors or 2 not in tensors:
        continue
    w0 = tensors[1]["w"]
    w1 = tensors[2]["w"]
    w0_can, _, _ = canonicalize_layer_tensor(tensors[1], Q_ORBITALS)
    w1_can, _, _ = canonicalize_layer_tensor(tensors[2], Q_ORBITALS)
    metrics_can = tensor_alignment_metrics(w0_can, w1_can)
    canonical_alignment_rows.append({
        "stage": stage,
        "layer_a": 1,
        "layer_b": 2,
        "relative_output_gauge_delta": metrics_can["relative_output_gauge_delta"],
        "projector_delta_per_sqrt_q": metrics_can["projector_delta_per_sqrt_q"],
        "principal_sval_min": metrics_can["principal_sval_min"],
        "principal_sval_mean": metrics_can["principal_sval_mean"],
        "principal_sval_max": metrics_can["principal_sval_max"],
    })
    if 3 in tensors:
        w2_can, _, _ = canonicalize_layer_tensor(tensors[3], Q_ORBITALS)
        metrics_p2 = tensor_alignment_metrics(w0_can, w2_can)
        period2_alignment_rows.append({
            "stage": stage,
            "layer_a": 1,
            "layer_b": 3,
            "relative_output_gauge_delta": metrics_p2["relative_output_gauge_delta"],
            "projector_delta_per_sqrt_q": metrics_p2["projector_delta_per_sqrt_q"],
            "principal_sval_min": metrics_p2["principal_sval_min"],
            "principal_sval_mean": metrics_p2["principal_sval_mean"],
            "principal_sval_max": metrics_p2["principal_sval_max"],
        })
    metrics = tensor_alignment_metrics(w0, w1)
    alignment_rows.append({
        "stage": stage,
        "layer_a": 1,
        "layer_b": 2,
        "relative_output_gauge_delta": metrics["relative_output_gauge_delta"],
        "projector_delta_per_sqrt_q": metrics["projector_delta_per_sqrt_q"],
        "principal_sval_min": metrics["principal_sval_min"],
        "principal_sval_mean": metrics["principal_sval_mean"],
        "principal_sval_max": metrics["principal_sval_max"],
    })
    w_fit = average_isometries(w0, w1)
    fit_df = run_repeated_candidate_flow(f"scale_fit_from_{stage}", w_fit, max_layers=MAX_LAYERS)
    fitted_rows.append(fit_df)

alignment_df = pd.DataFrame(alignment_rows)
canonical_alignment_df = pd.DataFrame(canonical_alignment_rows)
period2_alignment_df = pd.DataFrame(period2_alignment_rows)
fitted_summary_df = pd.concat(fitted_rows, ignore_index=True) if fitted_rows else pd.DataFrame()
combined_scale_summary_df = pd.concat([summary_df, fitted_summary_df], ignore_index=True) if not fitted_summary_df.empty else summary_df.copy()

alignment_path = DATA_DIR / "finite_depth_gaussian_mera_tensor_alignment.csv"
canonical_alignment_path = DATA_DIR / "finite_depth_gaussian_mera_canonical_tensor_alignment.csv"
period2_alignment_path = DATA_DIR / "finite_depth_gaussian_mera_period2_tensor_alignment.csv"
fitted_path = DATA_DIR / "finite_depth_gaussian_mera_scale_fit_summary.csv"
combined_path = DATA_DIR / "finite_depth_gaussian_mera_combined_scale_summary.csv"
alignment_df.to_csv(alignment_path, index=False)
canonical_alignment_df.to_csv(canonical_alignment_path, index=False)
period2_alignment_df.to_csv(period2_alignment_path, index=False)
fitted_summary_df.to_csv(fitted_path, index=False)
combined_scale_summary_df.to_csv(combined_path, index=False)

print(f"Saved tensor alignment: {alignment_path}")
print(f"Saved canonical tensor alignment: {canonical_alignment_path}")
print(f"Saved period-2 tensor alignment: {period2_alignment_path}")
print(f"Saved fitted scale-invariant summary: {fitted_path}")
display(alignment_df)
display(canonical_alignment_df)
if not period2_alignment_df.empty:
    display(period2_alignment_df)
display(fitted_summary_df)


## Reduced Domain-Wall-Channel Flow

In [ ]:

def edge_channel_basis(nx, dw_locs, width=EDGE_CHANNEL_WIDTH):
    xmask = domain_wall_x_mask(nx, dw_locs, width=width)
    cols = []
    labels = []
    q_full = 2 * int(nx)
    for x in np.where(xmask)[0]:
        for orb in range(2):
            vec = np.zeros(q_full, dtype=np.complex128)
            vec[2 * int(x) + int(orb)] = 1.0
            cols.append(vec)
            labels.append((int(x), int(orb)))
    basis = np.column_stack(cols) if cols else np.zeros((q_full, 0), dtype=np.complex128)
    return basis, labels


def repeated_row_basis(n_sites, basis):
    q_full, q_eff = basis.shape
    mat = np.zeros((int(n_sites) * q_full, int(n_sites) * q_eff), dtype=np.complex128)
    for y in range(int(n_sites)):
        mat[y*q_full:(y+1)*q_full, y*q_eff:(y+1)*q_eff] = basis
    return mat


def run_edge_channel_flow():
    basis, labels = edge_channel_basis(NX, DW_LOCS, width=EDGE_CHANNEL_WIDTH)
    q_eff = basis.shape[1]
    if q_eff == 0 or q_eff % 2:
        raise ValueError(f"Invalid edge-channel q_eff={q_eff}")
    Bglob = repeated_row_basis(NY, basis)
    C_edge_raw = Bglob.conj().T @ C0 @ Bglob
    C_edge_raw = 0.5 * (C_edge_raw + C_edge_raw.conj().T)
    C_edge, projected_evals = purify_to_half_filled_projector(C_edge_raw)
    rows = []
    occ_rows = []
    profile_rows = []
    C_current = C_edge
    C_raw_current = C_edge_raw
    n_sites = int(NY)
    embedding = Bglob.copy()
    for layer in range(0, int(MAX_LAYERS) + 1):
        entropy_data = mean_entropy_profile(C_current, n_sites, q_eff)
        fit = fit_entropy_logchord(entropy_data["Ay_values"], entropy_data["avg_entropy"], n_sites)
        x_profile = original_x_profile_from_embedding(embedding, NX, NY)
        rows.append({
            "smoke_test": bool(SMOKE_TEST),
            "stage": "edge_channel_uI_purified_initial",
            "layer": int(layer),
            "Nx": int(NX),
            "Ny_initial": int(NY),
            "Ny_current": int(n_sites),
            "q_eff": int(q_eff),
            "edge_width": int(EDGE_CHANNEL_WIDTH),
            "raw_idempotency_defect": idempotency_defect(C_raw_current),
            "measure_idempotency_defect": idempotency_defect(C_current),
            "slope": fit["slope"],
            "slope_err": fit["slope_err"],
            "c_est": fit["c_est"],
            "c_est_err": fit["c_est_err"],
            "r2": fit["r2"],
            "domain_wall_x_weight_width1": float(np.sum(x_profile[domain_wall_x_mask(NX, DW_LOCS)])),
        })
        for x, weight in enumerate(x_profile):
            profile_rows.append({"stage": "edge_channel_uI_purified_initial", "layer": int(layer), "x": int(x), "weight": float(weight)})
        if layer == int(MAX_LAYERS) or n_sites <= MIN_COARSE_NY:
            break
        data, C_tilde = extract_layer_isometry_after_u(C_current, n_sites, q_eff, None)
        C_raw, C_pure, raw_evals, embed_layer, _ = apply_layer(C_current, n_sites, q_eff, None, data["w"])
        for rank, val in enumerate(data["kept_occ"]):
            occ_rows.append({"layer": int(layer + 1), "sector": "kept", "rank": int(rank), "occupation": float(val), "chi": float(data["kept_chi"][rank])})
        for rank, val in enumerate(data["discarded_occ"]):
            occ_rows.append({"layer": int(layer + 1), "sector": "discarded", "rank": int(rank), "occupation": float(val), "chi": float(data["discarded_chi"][rank])})
        embedding = embedding @ embed_layer
        n_sites //= 2
        C_current = C_raw
        C_raw_current = C_raw
    return pd.DataFrame(rows), pd.DataFrame(occ_rows), pd.DataFrame(profile_rows), labels, projected_evals

edge_summary_df, edge_occupation_df, edge_profile_df, edge_labels, edge_projected_evals = run_edge_channel_flow()
edge_summary_path = DATA_DIR / "finite_depth_gaussian_mera_edge_channel_summary.csv"
edge_occupation_path = DATA_DIR / "finite_depth_gaussian_mera_edge_channel_occupations.csv"
edge_profile_path = DATA_DIR / "finite_depth_gaussian_mera_edge_channel_x_profiles.csv"
edge_summary_df.to_csv(edge_summary_path, index=False)
edge_occupation_df.to_csv(edge_occupation_path, index=False)
edge_profile_df.to_csv(edge_profile_path, index=False)
print(f"edge-channel labels = {edge_labels}")
print(f"Saved edge-channel summary: {edge_summary_path}")
display(edge_summary_df)


## Plots

In [ ]:
fig, ax = plt.subplots(figsize=(3.375, 2.45))
for stage, sub in summary_df.groupby("stage"):
    ax.errorbar(sub["layer"], sub["c_est"], yerr=sub["c_est_err"], marker="o", label=stage)
ax.axhline(1.0, color="k", ls="--", lw=1.0, label=r"$c_{\rm eff}=1$")
ax.set_xlabel("finite-depth layer")
ax.set_ylabel(r"$c_{\rm est}=3m$")
ax.set_title("Finite-depth entropy stability")
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "central_charge_finite_depth.png")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(3.375, 2.35))
for stage, sub in summary_df.groupby("stage"):
    ax.semilogy(sub["layer"], sub["raw_idempotency_defect"], marker="o", label=stage)
ax.set_xlabel("finite-depth layer")
ax.set_ylabel("raw idempotency defect")
ax.set_title("Raw Gaussian purity")
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "raw_idempotency_finite_depth.png")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(3.375, 2.35))
plot_df = optimizer_df.groupby(["stage", "layer"], as_index=False).tail(1)
for key in ["discarded_chi_mean", "discarded_chi_max", "cross", "idem_sq"]:
    ax.semilogy(plot_df["stage"] + ":L" + plot_df["layer"].astype(str), plot_df[key], marker="o", label=key)
ax.set_ylabel("component")
ax.set_title("Final layer-local diagnostics")
ax.tick_params(axis="x", rotation=35)
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "local_loss_components_finite_depth.png")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(3.375, 2.35))
for (stage, layer), sub in occupation_df.groupby(["stage", "layer"]):
    if layer != 1:
        continue
    for sector, ss in sub.groupby("sector"):
        alpha = 0.45 if sector == "discarded" else 0.75
        ax.plot(ss["rank"], ss["occupation"], marker="o", ms=2.5, lw=0.7, alpha=alpha, label=f"{stage} {sector}")
ax.axhline(0.5, color="k", ls="--", lw=0.8)
ax.set_xlabel("mode rank")
ax.set_ylabel(r"occupation $n_a$")
ax.set_title("Layer-1 kept/discarded spectra")
ax.legend(frameon=False, ncol=1)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "layer1_kept_discarded_spectra.png")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(8.7, 2.45), sharey=True)
stages = list(summary_df["stage"].drop_duplicates())
for ax, stage in zip(axes, stages):
    sub = profile_df[profile_df["stage"] == stage]
    for layer, ss in sub.groupby("layer"):
        ax.plot(ss["x"], ss["weight"], marker="o", ms=2.2, label=f"layer {layer}")
    for xdw in DW_LOCS:
        ax.axvline(int(xdw), color="k", ls="--", lw=0.8, alpha=0.6)
    ax.set_title(stage)
    ax.set_xlabel("original x")
axes[0].set_ylabel("retained-subspace weight")
axes[-1].legend(frameon=False)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "retained_x_profile_finite_depth.png")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(8.7, 2.45), sharey=True)
for ax, stage in zip(axes, stages):
    sub = spectrum_df[spectrum_df["stage"] == stage]
    for layer, ss in sub.groupby("layer"):
        vals = np.sort(ss["lambda"].to_numpy())
        near = vals[np.argsort(np.abs(vals - 0.5))[: min(24, len(vals))]]
        ax.scatter(np.full_like(near, layer, dtype=float), near, s=8, alpha=0.65, label=f"layer {layer}")
    ax.axhline(0.5, color="k", ls="--", lw=0.8)
    ax.set_title(stage)
    ax.set_xlabel("layer")
axes[0].set_ylabel(r"entanglement eigenvalue $\lambda$")
axes[-1].legend(frameon=False)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "entanglement_spectrum_finite_depth.png")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(3.375, 2.35))
for (stage, layer), sub in optimizer_df.groupby(["stage", "layer"]):
    if layer == 0:
        continue
    if "active" not in stage:
        continue
    ax.plot(sub["step"], sub["total"], marker="o", ms=2.2, label=f"{stage} L{layer}")
ax.set_yscale("log")
ax.set_xlabel("random active-u step")
ax.set_ylabel("best objective")
ax.set_title("Active disentangler search")
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "active_u_optimizer_finite_depth.png")
plt.show()

## Scale-Invariant Fit Diagnostics

In [ ]:

fig, ax = plt.subplots(figsize=(3.375, 2.35))
if not alignment_df.empty:
    x = np.arange(len(alignment_df))
    ax.bar(x - 0.18, alignment_df["relative_output_gauge_delta"], width=0.36, label="output-gauge delta")
    ax.bar(x + 0.18, alignment_df["projector_delta_per_sqrt_q"], width=0.36, label="projector delta")
    ax.set_xticks(x)
    ax.set_xticklabels(alignment_df["stage"], rotation=35, ha="right")
ax.set_ylabel("mismatch")
ax.set_title(r"Layer tensor comparison: $w_0$ vs $w_1$")
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "finite_depth_tensor_alignment.png")
plt.show()


In [ ]:

fig, ax = plt.subplots(figsize=(3.375, 2.45))
plot_summary = combined_scale_summary_df[
    combined_scale_summary_df["stage"].isin(["finite_depth_uI_raw"])
    | combined_scale_summary_df["stage"].str.startswith("scale_fit_from_finite_depth_uI_raw")
].copy()
for stage, sub in plot_summary.groupby("stage"):
    ax.errorbar(sub["layer"], sub["c_est"], yerr=sub["c_est_err"], marker="o", label=stage)
ax.axhline(1.0, color="k", ls="--", lw=1.0, label=r"$c_{\rm eff}=1$")
ax.set_xlabel("layer")
ax.set_ylabel(r"$c_{\rm est}=3m$")
ax.set_title("Finite-depth tensor vs averaged repeated fit")
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "finite_depth_vs_scale_fit_central_charge.png")
plt.show()


## Canonical Gauge and Edge-Channel Diagnostics

In [ ]:

fig, ax = plt.subplots(figsize=(3.375, 2.35))
if not canonical_alignment_df.empty:
    x = np.arange(len(canonical_alignment_df))
    ax.bar(x - 0.18, canonical_alignment_df["relative_output_gauge_delta"], width=0.36, label="canonical output delta")
    ax.bar(x + 0.18, canonical_alignment_df["projector_delta_per_sqrt_q"], width=0.36, label="canonical projector delta")
    ax.set_xticks(x)
    ax.set_xticklabels(canonical_alignment_df["stage"], rotation=35, ha="right")
ax.set_ylabel("mismatch")
ax.set_title("Canonical-gauge layer tensor comparison")
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "finite_depth_canonical_tensor_alignment.png")
plt.show()


In [ ]:

fig, ax = plt.subplots(figsize=(3.375, 2.45))
ax.errorbar(summary_df[summary_df["stage"] == "finite_depth_uI_raw"]["layer"],
            summary_df[summary_df["stage"] == "finite_depth_uI_raw"]["c_est"],
            yerr=summary_df[summary_df["stage"] == "finite_depth_uI_raw"]["c_est_err"],
            marker="o", label="full q finite-depth")
ax.errorbar(edge_summary_df["layer"], edge_summary_df["c_est"], yerr=edge_summary_df["c_est_err"], marker="s", label="DW-channel q_eff")
ax.axhline(1.0, color="k", ls="--", lw=1.0)
ax.set_xlabel("layer")
ax.set_ylabel(r"$c_{\rm est}=3m$")
ax.set_title("Full row basis vs reduced DW-channel basis")
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "edge_channel_central_charge.png")
plt.show()


In [ ]:

fig, ax = plt.subplots(figsize=(3.375, 2.35))
for layer, sub in edge_profile_df.groupby("layer"):
    ax.plot(sub["x"], sub["weight"], marker="o", ms=2.2, label=f"layer {layer}")
for xdw in DW_LOCS:
    ax.axvline(int(xdw), color="k", ls="--", lw=0.8, alpha=0.6)
ax.set_xlabel("original x")
ax.set_ylabel("retained-subspace weight")
ax.set_title("Reduced DW-channel retained profile")
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "edge_channel_retained_x_profile.png")
plt.show()


## Interpretation

In [ ]:
cols = [
    "stage", "layer", "Ny_current", "c_est", "c_est_err", "raw_idempotency_defect",
    "layer_discarded_chi_mean", "layer_discarded_chi_max", "domain_wall_x_weight_width1",
]
display(summary_df[[c for c in cols if c in summary_df.columns]])

if "alignment_df" in globals():
    display(alignment_df)
if "fitted_summary_df" in globals() and not fitted_summary_df.empty:
    display(fitted_summary_df[["stage", "layer", "Ny_current", "c_est", "c_est_err", "raw_idempotency_defect", "domain_wall_x_weight_width1"]])

if "canonical_alignment_df" in globals():
    display(canonical_alignment_df)
if "period2_alignment_df" in globals() and not period2_alignment_df.empty:
    display(period2_alignment_df)
if "edge_summary_df" in globals():
    display(edge_summary_df)
